# Subjecting codes of various dimensionality to global rotations

This tutorial reproduces **Fig. 4a** of [Bluvstein *et al.*, *Nature* **649**, 39 (2026)](https://www.nature.com/articles/s41586-025-09848-5), "A fault-tolerant neutral-atom architecture for universal quantum computation", using **Tsim**.

The experiment is simple to state. Take a logical qubit encoded in a quantum error-correcting code, prepare it in the logical $|+_L\rangle$ state, apply the *same* single-qubit $Z$-rotation $R_Z(\varphi)$ to every physical qubit at once, and read out the logical $X$ operator and the code stabilizers as a function of $\varphi$.

A global physical rotation acts as a **transversal logical gate**, and which logical gate you get depends on the *geometry* of the code. We will see that:

- a single unencoded qubit just traces out a smooth cosine,
- the **2D colour code** ([[7,1,3]] Steane code) shows flat *plateaus* at multiples of $90^\circ$ - the logical Clifford angles $S, S^\dagger, Z$,
- the **3D colour code** ([[15,1,3]] quantum Reed-Muller code) shows an *extra* plateau at multiples of $45^\circ$ - the angles $T, T^\dagger, TS, (TS)^\dagger$ of its transversal **non-Clifford $T$ gate**.

That $45^\circ$ plateau is the whole point. It is the gate that lifts a stabilizer code out of the Clifford group and makes universal, fault-tolerant computation possible. Tsim is a natural tool for this: a global $R_Z(\varphi)$ on the 15-qubit code is 15 non-Clifford rotations at once, which Tsim handles with ZX-calculus stabilizer-rank decomposition rather than a dense $2^{15}$ statevector.

## The physics

### Global rotations as transversal gates

The transversal operation here is a global $Z$-rotation,

$$U(\varphi) = \bigotimes_{j} R_Z(\varphi), \qquad R_Z(\varphi) = e^{-i\varphi Z/2}.$$

For a CSS code this commutes with every $Z$-type stabilizer, so the $Z$ checks are untouched; it does *not* commute with the $X$-type stabilizers or with $X_L$, and that is what we probe.

At special angles $U(\varphi)$ maps the code space onto itself and realizes a logical gate. At $\varphi = \pi/2$ this is always a logical Clifford. Whether $\varphi = \pi/4$ realizes a logical **$T$** depends on the code: it works exactly for codes whose $X$-stabilizers are *triorthogonal*. The [[15,1,3]] Reed-Muller code is built that way (it has a transversal $T$); the [[7,1,3]] Steane code is not (its $X$-checks have weight 4, only doubly-even), so it has no protected $45^\circ$ point.

The **Eastin-Knill theorem** forbids a transversal gate set that is both universal and unitary, which is why no single code plateaus everywhere. The paper circumvents this with logical measurement: a transversal CZ between a Reed-Muller $|T_L\rangle$ and $|+_L\rangle$ followed by measurement teleports an $H$, giving a universal $\{H, T, \text{CNOT}\}$ set from fully transversal operations.

### Why plateaus appear, and why stabilizer *signs* matter

Both codes have distance 3, so they correct any single-qubit error. Near an angle where $U(\varphi)$ is a clean logical gate, a small deviation $\delta\varphi$ looks like a low-weight coherent error the code absorbs, so the logical expectation stays *flat* - these are the plateaus - and the stabilizers *revive* toward $\pm 1$. A key subtlety from the paper: for the non-Clifford $T$ point, the stabilizers must have the **correct ($+1$) signs**. Flipping signs (here, the four corners of the Reed-Muller tetrahedron) destroys the $45^\circ$ plateau - the "3D negative stabilizer" control. This is the statement that **logical magic requires physical entanglement**: $|+_L\rangle$ is an eigenstate of a tensor-product operator $X_L = X_1X_2X_3\dots$, but $|T_L\rangle$ is an eigenstate of $\tfrac{1}{\sqrt2}(X_1X_2X_3\dots + Y_1Y_2Y_3\dots)$, a superposition spanning the code that is necessarily entangled.

### What Tsim computes

For each $\varphi$ Tsim builds the circuit as a ZX diagram, decomposes the non-Clifford rotations by stabilizer rank, and **samples** the readout. We use Tsim's sampler for the headline result (including the 15-rotation 3D code), and cross-check against an exact statevector reference computed from the Clifford encoder plus an analytic diagonal rotation.

In [ ]:
import numpy as np
import stim
import matplotlib.pyplot as plt

import tsim

## Building the codes

The [[7,1,3]], [[15,1,3]] and [[16,6,4]] codes in the paper are all members of the **quantum Reed-Muller family** built from the hypercube encoding circuit of Extended Data Fig. 10a (ref. 128). We will build that literal circuit below, but first we define the codes by their stabilizer groups - which is convenient and easy to verify:

- **[[7,1,3]] Steane / 2D colour code** - the self-dual CSS code of the $[7,4,3]$ Hamming code.
- **[[15,1,3]] Reed-Muller / 3D colour code** - $X$-checks from the punctured $\mathrm{RM}(1,4)$ code (weight-8), $Z$-checks from $\mathrm{RM}(2,4)$ (weight-8 and weight-4). This triorthogonal structure is exactly what gives the transversal $T$.

We ask `stim` for the Clifford that maps $|0\dots0\rangle$ onto the state stabilized by the generators together with $X_L$ (so the prepared state is the $+1$ eigenstate of $X_L$, i.e. $|+_L\rangle$), and verify it with `stim.Circuit.flow_generators()` as the issue suggests.

In [ ]:
def _pauli(kind, support, n):
    """Build a stim Pauli-string label, e.g. ('X', [0, 2], 4) -> 'X_X_'."""
    chars = ["_"] * n
    for q in support:
        chars[q] = kind
    return "".join(chars)


def plus_logical_encoder(n, x_stabs, z_stabs, x_logical, corner_flips=None):
    """Clifford circuit preparing |+_L> from |0...0>.

    The prepared state is the simultaneous +1 eigenstate of every stabilizer
    and of X_L. ``corner_flips`` appends a final X(pi) pulse to the listed
    qubits: flipping the four corners of the Reed-Muller tetrahedron gives the
    'incorrect stabilizer signs' of the paper's '3D negative stabilizer'
    control, which destroys the transversal-T plateau at 45 degrees.
    """
    generators = [stim.PauliString(_pauli("X", s, n)) for s in x_stabs]
    generators += [stim.PauliString(_pauli("Z", s, n)) for s in z_stabs]
    generators.append(stim.PauliString(_pauli("X", x_logical, n)))

    tableau = stim.Tableau.from_stabilizers(
        generators, allow_redundant=False, allow_underconstrained=False
    )
    circuit = tableau.to_circuit("elimination")
    for q in corner_flips or []:
        circuit.append("X", [q])
    return circuit

In [ ]:
# --- [[7,1,3]] Steane code (2D colour code) ----------------------------------
# Self-dual: X and Z checks share the [7,4,3] Hamming support.
steane = dict(
    n=7,
    x_stabs=[[3, 4, 5, 6], [1, 2, 5, 6], [0, 2, 4, 6]],
    z_stabs=[[3, 4, 5, 6], [1, 2, 5, 6], [0, 2, 4, 6]],
    x_logical=list(range(7)),  # transversal all-X representative of X_L
)

# --- [[15,1,3]] quantum Reed-Muller code (3D colour code) --------------------
# Qubit j (0-indexed) sits at the 4-bit point (j + 1) of the punctured tesseract.
# X-checks = punctured RM(1,4) (the four coordinate functions, weight 8).
# Z-checks = RM(2,4) (those four, plus the six degree-2 monomials, weight 4).
def _bit(j, k):
    return (j >> k) & 1


_rm_x = [[j for j in range(15) if _bit(j + 1, k)] for k in range(4)]
_rm_z = [list(s) for s in _rm_x] + [
    [j for j in range(15) if _bit(j + 1, a) and _bit(j + 1, b)]
    for a in range(4)
    for b in range(a + 1, 4)
]
# The four corners of the tetrahedron are the weight-1 points e_k (1,2,4,8),
# i.e. qubits 0, 1, 3, 7. Flipping them sets incorrect stabilizer signs.
reed_muller = dict(
    n=15, x_stabs=_rm_x, z_stabs=_rm_z, x_logical=list(range(15)), corners=[0, 1, 3, 7]
)

# --- Unentangled reference: 15 bare qubits "analysed as a 3D code" -----------
# As in the paper, 15 unentangled physical qubits in |+>^15 are read out with
# the *same* Reed-Muller operators (X_L and the weight-8 X-checks), so the
# only difference from the 3D curve is the missing entanglement.
unentangled = dict(
    n=15, x_stabs=_rm_x, z_stabs=[], x_logical=list(range(15)), product=True
)

print("Steane X-stab weights :", [len(s) for s in steane["x_stabs"]])
print("RM     X-stab weights :", [len(s) for s in reed_muller["x_stabs"]])
print("RM     Z-stab weights :", [len(s) for s in reed_muller["z_stabs"]])

`flow_generators()` reports how the Clifford encoder propagates the input Paulis. Below, the input $Z_{14}$ flows to the all-$X$ logical $X_L$ and the inputs $X_i$ flow to the code stabilizers - confirming we prepared the intended $|+_L\rangle$ of the [[15,1,3]] code.

In [ ]:
encoder = plus_logical_encoder(
    reed_muller["n"], reed_muller["x_stabs"], reed_muller["z_stabs"], reed_muller["x_logical"]
)
print(f"[[15,1,3]] encoder: {len(encoder)} gates, {len(encoder.flow_generators())} flow generators")
for flow in encoder.flow_generators()[:6]:
    print("   ", flow)
print("    ...")

## The hypercube encoding circuit (Extended Data Fig. 10a)

The paper prepares these codes with a single *hypercube encoding circuit* (ref. 128): place $2^m$ qubits at the corners of an $m$-cube, apply the transversal CNOT network $E = \bigl(\begin{smallmatrix}1&0\\1&1\end{smallmatrix}\bigr)^{\otimes m}$ (at step $t$, a CNOT across every edge in direction $t$), with a programmable product input - $|+\rangle$ on rows of low Hamming weight, $|0\rangle$ on rows of high weight. A different input pattern (set by local $Y(\pi/2)$ pulses) gives a different code from the *same* entangling structure:

- **[[16,6,4]]** tesseract - full $m=4$ cube, no puncture;
- **[[15,1,3]]** - $m=4$ cube punctured at the $0\dots0$ corner;
- **[[7,1,3]]** - $m=3$ cube punctured at the $0\dots0$ corner.

We build this literal circuit and check, by statevector overlap, that it prepares exactly the same $|+_L\rangle$ as the stabilizer encoder above. (For the rotation sweep we then use the deterministic stabilizer encoder, since it prepares the identical state without the puncturing measurement.)

In [ ]:
def hypercube_plus_logical(m, r_x):
    """Prepare |+_L> via the paper's hypercube encoder on an m-cube.

    |+> on rows of Hamming weight <= r_x (and on the logical corner 0...0),
    |0> elsewhere, then the transversal CNOT network E; the 0...0 corner is
    punctured by projecting it onto |+> (an X-basis logical measurement).
    Returns the statevector of the remaining 2^m - 1 physical qubits.
    """
    n = 1 << m
    circuit = stim.Circuit()
    for q in range(n):
        if q == 0 or bin(q).count("1") <= r_x:
            circuit.append("H", [q])  # |+> input; all others stay |0>
    for t in range(m):  # E = ((1,0),(1,1))^{otimes m}: CNOT across edge-direction t
        for q in range(n):
            if _bit(q, t) == 0:
                circuit.append("CX", [q, q | (1 << t)])
    sim = stim.TableauSimulator()
    sim.do_circuit(circuit)
    psi = np.asarray(sim.state_vector(endian="big"), dtype=np.complex128).reshape([2] * n)
    branch0 = psi[(0, *([slice(None)] * (n - 1)))].reshape(-1)
    branch1 = psi[(1, *([slice(None)] * (n - 1)))].reshape(-1)
    sub = branch0 + branch1  # project the punctured corner onto |+>
    return sub / np.linalg.norm(sub)


def _stabilizer_statevector(code):
    sim = stim.TableauSimulator()
    sim.do_circuit(plus_logical_encoder(code["n"], code["x_stabs"], code["z_stabs"], code["x_logical"]))
    return np.asarray(sim.state_vector(endian="big"), dtype=np.complex128)


overlap_steane = abs(np.vdot(hypercube_plus_logical(3, 1), _stabilizer_statevector(steane)))
overlap_rm = abs(np.vdot(hypercube_plus_logical(4, 1), _stabilizer_statevector(reed_muller)))
print(f"hypercube vs stabilizer encoder, |<psi|psi>|:")
print(f"  [[7,1,3]]  (3-cube): {overlap_steane:.6f}")
print(f"  [[15,1,3]] (4-cube): {overlap_rm:.6f}")
assert overlap_steane > 1 - 1e-6 and overlap_rm > 1 - 1e-6
print("  -> the literal hypercube circuit prepares the same |+_L> state.")

## The rotation experiment

For a given angle $\varphi$ the circuit is: encode $|+_L\rangle$, apply $R_Z(\varphi)$ to every physical qubit, then read out $X_L$ and the $X$-stabilizers. Tsim takes angles in units of $\pi$, so $\varphi$ radians is `R_Z(phi / pi)`. We read out with `MPP` (Pauli-product measurement) on the logical operator and each $X$-stabilizer, tagged with `OBSERVABLE_INCLUDE` and `DETECTOR`; measuring the products directly keeps the sampler's output count small, which matters at 15 qubits.

Tsim also shows *why* the 3D case is the hard one: `tcount()` reports the number of non-Clifford gates.

In [ ]:
def rotated_circuit(code, phi, *, corner_flips=None):
    """|+_L>, a global R_Z(phi) (phi in radians), then MPP readout of X_L and
    the X-stabilizers, annotated for Tsim's detector sampler.
    """
    n = code["n"]
    enc = plus_logical_encoder(
        code["n"], code["x_stabs"], code["z_stabs"], code["x_logical"], corner_flips
    )
    text = str(enc) + "\n"
    text += f"R_Z({phi / np.pi}) " + " ".join(map(str, range(n))) + "\n"
    # rec[-(n_stab + 1)] is X_L; the last n_stab records are the stabilizers.
    text += "MPP " + "*".join(f"X{q}" for q in code["x_logical"]) + "\n"
    for sup in code["x_stabs"]:
        text += "MPP " + "*".join(f"X{q}" for q in sup) + "\n"
    n_stab = len(code["x_stabs"])
    text += f"OBSERVABLE_INCLUDE(0) rec[{-(n_stab + 1)}]\n"
    for i in range(n_stab):
        text += f"DETECTOR rec[{-(n_stab - i)}]\n"
    return tsim.Circuit(text)


# At phi = pi/4 the global rotation is a transversal T: one T gate per qubit.
print("non-Clifford gate count (tcount) at phi = pi/4:")
print("  [[7,1,3]] :", rotated_circuit(steane, np.pi / 4).tcount())
print("  [[15,1,3]]:", rotated_circuit(reed_muller, np.pi / 4).tcount())

In [ ]:
# Tsim's ZX timeline view of the Steane circuit at phi = pi/4.
rotated_circuit(steane, np.pi / 4).diagram("timeline-svg", height=260)

### Exact reference values

For ground truth we compute the expectation values exactly. The encoder is Clifford, so `stim`'s tableau simulator gives $|+_L\rangle$ as a statevector; the global rotation is then applied analytically, multiplying the amplitude of a computational basis state $|b\rangle$ by $e^{i\varphi\,\mathrm{popcount}(b)}$ (up to a global phase). An $X$-type operator flips the bits on its support, so $\langle X_S\rangle = \mathrm{Re}\,\langle\psi|\psi_{\text{flip}(S)}\rangle$.

This is exact and cheap (one $2^n$ vector reused for every angle). It is *not* how Tsim or the paper work at scale - a dense $2^{15}$ contraction of the rotated diagram exhausts memory - but it is the right yardstick for these small codes.

In [ ]:
def _support_mask(support, n):
    """Bitmask of an operator support; qubit q occupies bit (n - 1 - q)."""
    mask = 0
    for q in support:
        mask |= 1 << (n - 1 - q)
    return mask


def _rotated_statevector(code, phi, corner_flips=None):
    if code.get("product"):
        # Unentangled reference: |+>^n product state, no encoder.
        prep = stim.Circuit("H " + " ".join(map(str, range(code["n"]))))
    else:
        prep = plus_logical_encoder(
            code["n"], code["x_stabs"], code["z_stabs"], code["x_logical"], corner_flips
        )
    sim = stim.TableauSimulator()
    sim.do_circuit(prep)
    psi0 = np.asarray(sim.state_vector(endian="big"), dtype=np.complex128)
    idx = np.arange(psi0.size)
    popcount = np.array([int(b).bit_count() for b in idx])
    return psi0 * np.exp(1j * phi * popcount), idx


def exact_curves(code, phis, *, corner_flips=None):
    """Raw (undecoded) <X_L>(phi) and the mean |<X-stabilizer>|(phi)."""
    n = code["n"]
    mask_logical = _support_mask(code["x_logical"], n)
    mask_stabs = [_support_mask(s, n) for s in code["x_stabs"]]
    logical, stabilizer = [], []
    for phi in phis:
        psi, idx = _rotated_statevector(code, phi, corner_flips)
        logical.append(float(np.real(np.vdot(psi, psi[idx ^ mask_logical]))))
        if mask_stabs:
            stabilizer.append(
                float(np.mean([abs(np.real(np.vdot(psi, psi[idx ^ m]))) for m in mask_stabs]))
            )
        else:
            stabilizer.append(np.nan)
    return np.array(logical), np.array(stabilizer)


# Sanity check: at phi = 0 the state is |+_L>, so every expectation value is +1.
for name, code in [("steane", steane), ("reed_muller", reed_muller)]:
    xl0, s0 = exact_curves(code, [0.0])
    assert np.isclose(xl0[0], 1.0, atol=1e-6), (name, xl0)
    assert np.isclose(s0[0], 1.0, atol=1e-6), (name, s0)
print("phi = 0 self-check passed: <X_L> = <stabilizers> = +1 for both codes.")

### Sampling with Tsim

Now the part that actually needs Tsim: turning a circuit with many non-Clifford rotations into samples. We compile a detector sampler and estimate each expectation value from Monte-Carlo shots via $\langle P\rangle = 1 - 2\,\Pr(\text{outcome} = 1)$.

In [ ]:
def sampled_curves(code, phis, shots, *, corner_flips=None, seed=1, postselect=False):
    """Estimate <X_L>(phi) and mean |<X-stabilizer>|(phi) with Tsim's sampler.

    With ``postselect=True`` the logical estimate keeps only shots whose
    syndrome is trivial (all detectors = 0), matching the decoded/postselected
    top panel of Fig. 4a.
    """
    logical, stabilizer = [], []
    for phi in phis:
        circuit = rotated_circuit(code, phi, corner_flips=corner_flips)
        sampler = circuit.compile_detector_sampler(seed=seed)
        det, obs = sampler.sample(shots=shots, separate_observables=True)
        if postselect and code["x_stabs"]:
            accepted = obs[~det.any(axis=1)]
            logical.append(1.0 - 2.0 * accepted.mean() if accepted.size else np.nan)
        else:
            logical.append(1.0 - 2.0 * obs.mean())
        if code["x_stabs"]:
            stabilizer.append(float(np.mean(np.abs(1.0 - 2.0 * det.mean(axis=0)))))
        else:
            stabilizer.append(np.nan)
    return np.array(logical), np.array(stabilizer)

Before the full sweep, confirm the headline claim: **Tsim's sampler runs the 3D [[15,1,3]] code**, where a generic angle is 15 non-Clifford rotations at once. We check a few angles against the exact values, including a generic $30^\circ$ (the genuinely hard case) and $45^\circ$ (15 transversal $T$ gates).

In [ ]:
check_phis = np.deg2rad([0, 30, 45, 90])
xl_exact, _ = exact_curves(reed_muller, check_phis)
xl_sampled, _ = sampled_curves(reed_muller, check_phis, shots=8_000)
print("  phi      exact     Tsim")
for phi, e, s in zip(np.rad2deg(check_phis), xl_exact, xl_sampled):
    print(f"  {phi:5.1f}   {e:+.3f}   {s:+.3f}")

### Postselection and acceptance fraction

The paper sharpens its plateaus with *postselection*, accepting only shots whose stabilizers are satisfied (it quotes acceptance fractions of 46% / 74% for the 3D / 2D codes). We implement the same idea: project the rotated state onto the $+1$ eigenspace of the $X$-stabilizers and report the accepted $\langle X_L\rangle$ and the acceptance fraction. On *noiseless* data the curves already lie almost entirely in the code space, so postselection barely moves them and the acceptance stays high; with physical noise it is what removes the off-codespace weight (alongside the paper's purity normalization).

In [ ]:
def postselected_logical(code, phi, corner_flips=None):
    """<X_L> conditioned on all X-stabilizers = +1, and the acceptance fraction."""
    n = code["n"]
    psi, idx = _rotated_statevector(code, phi, corner_flips)
    for sup in code["x_stabs"]:  # project onto +1 eigenspace of each X-stabilizer
        psi = 0.5 * (psi + psi[idx ^ _support_mask(sup, n)])
    accept = float(np.real(np.vdot(psi, psi)))
    if accept < 1e-12:
        return 0.0, 0.0
    xl = float(np.real(np.vdot(psi, psi[idx ^ _support_mask(code["x_logical"], n)])) / accept)
    return xl, accept


print("  phi    raw <X_L>   postsel <X_L>   acceptance")
for deg in (0, 22.5, 30, 45):
    raw, _ = exact_curves(reed_muller, [np.deg2rad(deg)])
    xl_ps, acc = postselected_logical(reed_muller, np.deg2rad(deg))
    print(f"  {deg:5.1f}    {raw[0]:+.3f}       {xl_ps:+.3f}        {acc:.3f}")

## Sweeping the global angle

Now we sweep $\varphi$ from $-180^\circ$ to $180^\circ$ for all four configurations of Fig. 4a:

- **3D colour** - the [[15,1,3]] Reed-Muller code,
- **2D colour** - the [[7,1,3]] Steane code,
- **Unentangled** - 15 bare qubits in $|+\rangle^{\otimes 15}$, read out with the same Reed-Muller operators,
- **3D negative stabilizer** - the Reed-Muller code with the four tetrahedron corners flipped (the paper's control: incorrect stabilizer signs, which should *remove* the $45^\circ$ plateau).

Following the paper's analysis, the two *encoded* curves show the **decoded** logical projection - $\langle X_L\rangle$ postselected on a satisfied syndrome - which turns the oscillating raw expectation of a many-qubit code into the flat transversal-gate plateaus. The unentangled and negative-stabilizer controls are shown **raw**: with no (correctly signed) code space to project onto, $\langle X_L\rangle = \cos^{15}\varphi$ collapses away from $0^\circ$ and $\pm180^\circ$, and the two control curves hug each other - exactly the behaviour that distinguishes a genuine code state from its look-alikes. The bottom panel shows the raw $X$-stabilizer revival for all four. The open markers are Tsim samples (syndrome-postselected, on the 2D curve and on the 3D code at the multiples of $45^\circ$). Raise `SAMPLE_SHOTS` for tighter error bars.

In [ ]:
phi_deg = np.linspace(-180, 180, 181)
phi_rad = np.deg2rad(phi_deg)
phi_dots = np.linspace(-180, 180, 73)  # marker spacing, as in the figure
phi_dots_rad = np.deg2rad(phi_dots)

# Top panel, as in Fig. 4a:
# - 3D and 2D codes: the *decoded* logical projection (postselected on a
#   satisfied syndrome). The raw <X_L> of a many-qubit code is multi-frequency
#   and oscillates; projecting onto the code space recovers the flat
#   transversal-gate plateaus.
# - Unentangled and 3D-negative-stabilizer: the *raw* projection. With no
#   (correct) code space to project onto, <X_L> = cos(phi)^15 collapses to
#   zero away from 0 and +-180 degrees -- these two curves hug each other.
xl_3d = np.array([postselected_logical(reed_muller, p)[0] for p in phi_rad])
xl_2d = np.array([postselected_logical(steane, p)[0] for p in phi_rad])
xl_un = exact_curves(unentangled, phi_rad)[0]
xl_neg = exact_curves(reed_muller, phi_rad, corner_flips=reed_muller["corners"])[0]

dots_3d = np.array([postselected_logical(reed_muller, p)[0] for p in phi_dots_rad])
dots_2d = np.array([postselected_logical(steane, p)[0] for p in phi_dots_rad])
dots_un = exact_curves(unentangled, phi_dots_rad)[0]
dots_neg = exact_curves(reed_muller, phi_dots_rad, corner_flips=reed_muller["corners"])[0]

# Bottom panel: the raw X-stabilizer revival for all four configurations.
st_3d = exact_curves(reed_muller, phi_rad)[1]
st_2d = exact_curves(steane, phi_rad)[1]
st_un = exact_curves(unentangled, phi_rad)[1]
st_neg = exact_curves(reed_muller, phi_rad, corner_flips=reed_muller["corners"])[1]
st_dots_3d = exact_curves(reed_muller, phi_dots_rad)[1]
st_dots_2d = exact_curves(steane, phi_dots_rad)[1]
st_dots_un = exact_curves(unentangled, phi_dots_rad)[1]
st_dots_neg = exact_curves(reed_muller, phi_dots_rad, corner_flips=reed_muller["corners"])[1]

# Tsim sampled overlay, postselected on a trivial syndrome to match the top panel.
SAMPLE_SHOTS = 8_000
phi_marks_2d = np.deg2rad(np.linspace(-180, 180, 13))
phi_marks_3d = np.deg2rad(np.arange(-180, 181, 45))  # Clifford + T angles: cheap to compile
xl_2d_s, _ = sampled_curves(steane, phi_marks_2d, shots=SAMPLE_SHOTS, postselect=True)
xl_3d_s, _ = sampled_curves(reed_muller, phi_marks_3d, shots=SAMPLE_SHOTS, postselect=True)

In [ ]:
def _unit(curve):
    """Normalize to unity maximum expectation, as in Fig. 4a (Methods)."""
    peak = np.nanmax(np.abs(curve))
    return curve / peak if peak > 0 else curve


C_3D, C_2D, C_UN, C_NEG = "#c0392b", "#1f3a6b", "#4f8a4f", "#b9b9b9"
gate_labels = ["Z", "(TS)†", "S†", "T†", "I", "T", "S", "TS", "Z"]
gate_ticks = np.arange(-180, 181, 45)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(8.2, 5.4), sharex=True, gridspec_kw=dict(height_ratios=[1.55, 1])
)


def _curve(ax, x_line, y_line, x_dots, y_dots, color, label=None):
    ax.plot(x_line, _unit(y_line), color=color, lw=1.1, zorder=1)
    ax.plot(
        x_dots, _unit(y_dots), "o", color=color, ms=3.2,
        mfc=color, mec="white", mew=0.4, label=label, zorder=2,
    )


# Top panel: logical X projection.
_curve(ax_top, phi_deg, xl_3d, phi_dots, dots_3d, C_3D, "3D colour")
_curve(ax_top, phi_deg, xl_2d, phi_dots, dots_2d, C_2D, "2D colour")
_curve(ax_top, phi_deg, xl_un, phi_dots, dots_un, C_UN, "Unentangled")
_curve(ax_top, phi_deg, xl_neg, phi_dots, dots_neg, C_NEG, "3D negative\nstabilizer")
ax_top.plot(np.rad2deg(phi_marks_3d), xl_3d_s, "o", color=C_3D, ms=5.5, mfc="white", label="3D (Tsim)")
ax_top.plot(np.rad2deg(phi_marks_2d), xl_2d_s, "s", color=C_2D, ms=4, mfc="white", label="2D (Tsim)")
ax_top.set_ylabel("Logical X\nprojection")
ax_top.set_ylim(-1.06, 1.06)
ax_top.set_yticks([-1, 0, 1])
ax_top.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8.5, handlelength=1.1, frameon=False)
for tick, label in zip(gate_ticks, gate_labels):
    ax_top.text(tick, 1.13, label, ha="center", fontsize=9)

# Bottom panel: normalized absolute stabilizer, all four configurations.
_curve(ax_bot, phi_deg, st_3d, phi_dots, st_dots_3d, C_3D)
_curve(ax_bot, phi_deg, st_2d, phi_dots, st_dots_2d, C_2D)
_curve(ax_bot, phi_deg, st_un, phi_dots, st_dots_un, C_UN)
_curve(ax_bot, phi_deg, st_neg, phi_dots, st_dots_neg, C_NEG)
ax_bot.set_ylabel("Normalized abs.\nstabilizer")
ax_bot.set_ylim(-0.03, 1.03)
ax_bot.set_yticks([0, 0.5, 1.0])
ax_bot.set_xlabel(r"Global $\varphi$")
ax_bot.set_xticks(range(-180, 181, 90))
ax_bot.set_xticklabels(["-180°", "-90°", "0", "90°", "180°"])

fig.subplots_adjust(hspace=0.08)
plt.show()

## Reading the figure

The top panel is the logical $X$ projection; the bottom panel is the (normalized) magnitude of the $X$-stabilizers. The labels along the top are the logical gate realized at each angle.

- **Unentangled (green).** Fifteen bare qubits read out with the code's operators: $\langle X_L\rangle = \cos^{15}\varphi$ collapses to zero away from $0^\circ$ and $\pm180^\circ$, and the weight-8 stabilizers $\langle X_S\rangle = \cos^{8}\varphi$ revive only at multiples of $180^\circ$. No entanglement, no protection.
- **2D colour / Steane (blue).** Decoded plateaus at multiples of $90^\circ$, where the global rotation realizes a logical Clifford ($S, S^\dagger, Z$); the stabilizers revive only there (dipping to $1/2$ in between).
- **3D colour / Reed-Muller (red).** An *extra* plateau at $\pm 45^\circ$ and $\pm 135^\circ$ - the transversal **$T, T^\dagger, TS, (TS)^\dagger$** gates - with the stabilizers reviving there too. This is exactly the feature the experiment uses for universality.
- **3D negative stabilizer (grey).** The same code with the four tetrahedron corners flipped. Its raw projection hugs the *unentangled* curve: with incorrect stabilizer signs there is no $45^\circ$ plateau and no protection, even though the connectivity and gate count are identical to the red curve. Correct stabilizer signs - i.e. the right entangled state - are essential for the non-Clifford gate.

The open Tsim markers sit on the decoded curves, including the red circles at $\pm 45^\circ$ - the points where the 15-qubit code is running 15 non-Clifford gates at once. Reproducing those is the heart of Fig. 4a, and is exactly the regime where Tsim's stabilizer-rank sampler is needed (a dense statevector of the rotated diagram is out of reach).

### Why this matters

The $45^\circ$ plateau is the fingerprint of a transversal non-Clifford gate, and it only exists when the code carries genuine in-block entanglement - $|T_L\rangle$ is a code-spanning superposition, whereas a logical Pauli is a product operator. In the paper this is the basis for universal logic via transversal teleportation between 2D and 3D codes, and it underlies an error-corrected Bell (CHSH) test that reaches $1.99(3)\times\sqrt2$, linking quantum contextuality to the entanglement that error correction must protect.

### Things to try

- Increase `SAMPLE_SHOTS` and the number of sampled markers and watch the Monte-Carlo points tighten onto the exact curves.
- Flip a different set of qubits via `corner_flips`, or rotate about another axis (`R_X`, `R_Y`), and watch the plateaus move.
- Build the high-rate [[16,6,4]] tesseract code with the hypercube encoder *without* puncturing (the full 4-cube) and study one of its six logical qubits.